# Lecture 4 · Edge Detection
> **MCS 3950 Computer Vision**  
> Follow this notebook alongside the L4 slides. Run every cell in order before moving to the next section.

| Section | Topic | Approx. time |
|---------|-------|-------------|
| **0** | Image gradients from scratch | 15 min |
| **1** | Sobel & Prewitt operators | 10 min |
| **2** | Laplacian of Gaussian (LoG) | 10 min |
| **3** | The Canny edge detector | 15 min |
| **4** | Detector comparison | 10 min |
| **Try at Home** | Exercises A & B | — |

**Images used:** `skimage.data.camera()` (grayscale), `skimage.data.chelsea()` (colour)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage import data, filters, feature
from scipy.ndimage import convolve, correlate

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
})

# ── Load images ──────────────────────────────────────────────────────────
camera  = data.camera()          # 512×512 uint8 grayscale
chelsea = data.chelsea()          # 300×451×3 uint8 RGB
chelsea_gray = cv2.cvtColor(chelsea, cv2.COLOR_RGB2GRAY)

print(f'camera:  {camera.shape}  dtype={camera.dtype}')
print(f'chelsea: {chelsea.shape}  dtype={chelsea.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(camera,  cmap='gray'); axes[0].set_title('camera (grayscale)'); axes[0].axis('off')
axes[1].imshow(chelsea);              axes[1].set_title('chelsea (colour)');    axes[1].axis('off')
plt.tight_layout(); plt.show()

---
## Section 0 — Image Gradients from Scratch

Before using any library, let's implement gradient computation manually to understand exactly what is happening.

### 0.1 — 1-D Signal: Visualising the Derivative

A clean step edge produces a spike in the first derivative (gradient) and a zero-crossing in the second derivative (Laplacian).

In [ ]:
x = np.arange(100)
# Step edge at x=50, ramp from 40 to 200
signal = np.where(x < 50, 40.0, 200.0).astype(float)
signal_noisy = signal + np.random.default_rng(42).normal(0, 20, size=100)

# First derivative (central difference)
d1_clean = np.gradient(signal)
d1_noisy = np.gradient(signal_noisy)

# Second derivative
d2_clean = np.gradient(d1_clean)
d2_noisy = np.gradient(d1_noisy)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
titles = ['Signal (clean)', 'First derivative |∂f/∂x|', 'Second derivative ∂²f/∂x²',
          'Signal (noisy)',  'First derivative — noisy',  'Second derivative — noisy']
data_pairs = [signal, np.abs(d1_clean), d2_clean,
              signal_noisy, np.abs(d1_noisy), d2_noisy]
colours = ['royalblue','forestgreen','darkorange'] * 2
for ax, d, t, c in zip(axes.flat, data_pairs, titles, colours):
    ax.plot(x, d, color=c, linewidth=1.5)
    ax.axvline(50, color='red', linestyle='--', linewidth=1, label='edge x=50')
    ax.set_title(t); ax.legend(fontsize=8)
plt.suptitle('Edge = spike in 1st derivative  /  zero-crossing in 2nd derivative', fontsize=12)
plt.tight_layout(); plt.show()

### 0.2 — 2-D Gradient: Finite Differences on an Image

We implement **central differences** manually using NumPy slicing.

In [ ]:
def gradient_2d(img):
    """Central-difference gradient. Returns Gx, Gy (float64).
    img: 2-D array (H, W).
    """
    img = img.astype(float)
    # Horizontal gradient (x-direction = columns)
    Gx = np.zeros_like(img)
    Gx[:, 1:-1] = (img[:, 2:] - img[:, :-2]) / 2   # central
    Gx[:, 0]    =  img[:, 1]  - img[:, 0]           # forward at left edge
    Gx[:, -1]   =  img[:, -1] - img[:, -2]          # backward at right edge

    # Vertical gradient (y-direction = rows)
    Gy = np.zeros_like(img)
    Gy[1:-1, :] = (img[2:, :] - img[:-2, :]) / 2   # central
    Gy[0, :]    =  img[1, :]  - img[0, :]           # forward at top
    Gy[-1, :]   =  img[-1, :] - img[-2, :]          # backward at bottom

    return Gx, Gy

Gx, Gy = gradient_2d(camera)
magnitude  = np.hypot(Gx, Gy)          # |∇f| = sqrt(Gx² + Gy²)
direction  = np.arctan2(Gy, Gx)        # θ ∈ [−π, π]

def norm255(arr):
    """Normalise array to 0-255 uint8 for display.""",
    lo, hi = arr.min(), arr.max()
    return ((arr - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
show_data = [camera, norm255(Gx), norm255(Gy), norm255(magnitude)]
titles    = ['Original', 'Gx  (∂f/∂x)', 'Gy  (∂f/∂y)', 'Magnitude |∇f|']
for ax, img, t in zip(axes, show_data, titles):
    ax.imshow(img, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Central-difference gradient components', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Visualise gradient direction with HSV: hue=direction, value=magnitude
magnitude_norm = (magnitude / magnitude.max()).astype(np.float32)
direction_norm = ((direction + np.pi) / (2 * np.pi)).astype(np.float32)  # 0-1

hsv = np.zeros((*camera.shape, 3), dtype=np.float32)
hsv[..., 0] = direction_norm   # hue = gradient direction
hsv[..., 1] = 1.0              # full saturation
hsv[..., 2] = magnitude_norm   # value = magnitude (dark = weak gradient)

rgb = cv2.cvtColor((hsv * 255).astype(np.uint8), cv2.COLOR_HSV2RGB)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(norm255(magnitude), cmap='hot'); axes[0].set_title('Gradient magnitude  |∇f|'); axes[0].axis('off')
axes[1].imshow(rgb);                            axes[1].set_title('Gradient direction  θ  (colour = angle)'); axes[1].axis('off')
plt.suptitle('Colour encodes direction — bright = strong gradient', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 1 — Sobel & Prewitt Operators

Both operators combine smoothing (in one direction) with differentiation (in the other), making them more noise-robust than pure finite differences.

In [ ]:
# Sobel kernels
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=float)

sobel_y = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]], dtype=float)

# Two things must match for this to agree with OpenCV:
#
#   1. Correlation vs convolution. cv2.Sobel/filter2D CORRELATE — they slide
#      the kernel as written. scipy's convolve() does true convolution, which
#      flips the kernel 180 degrees; for an antisymmetric kernel like Sobel
#      that simply negates the result. np.hypot hides it, but the signed Gx
#      image comes out as a photographic negative. Use correlate().
#
#   2. Border handling. scipy's mode='reflect' duplicates the edge pixel
#      (d c b a | a b c d) = cv2's BORDER_REFLECT. cv2's *default* is
#      BORDER_REFLECT_101 (d c b | a b c d), which does not. scipy's name
#      for that is mode='mirror'. Only the 1-pixel frame differs — but that
#      is enough to blow up a .max() comparison.
Sx = correlate(camera.astype(float), sobel_x, mode='mirror')
Sy = correlate(camera.astype(float), sobel_y, mode='mirror')
Smag = np.hypot(Sx, Sy)

# Compare to cv2.Sobel
cv_Sx = cv2.Sobel(camera, cv2.CV_64F, 1, 0, ksize=3)
cv_Sy = cv2.Sobel(camera, cv2.CV_64F, 0, 1, ksize=3)
cv_Smag = np.hypot(cv_Sx, cv_Sy)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pairs = [(camera, 'Original'), (norm255(Sx), 'Sobel Gx (manual)'),
         (norm255(Sy), 'Sobel Gy (manual)'), (norm255(Smag), 'Sobel |G| (manual)'),
         (camera, 'Original'), (norm255(cv_Sx), 'Sobel Gx (cv2)'),
         (norm255(cv_Sy), 'Sobel Gy (cv2)'), (norm255(cv_Smag), 'Sobel |G| (cv2)')]
for ax, (img, title) in zip(axes.flat, pairs):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.suptitle('Manual vs cv2 Sobel — should be nearly identical', fontsize=12)
plt.tight_layout(); plt.show()

# Verify agreement
diff = np.abs(Smag - cv_Smag).max()
print(f'Max pixel difference (manual vs cv2): {diff:.4f}  (should be ~0)')

In [ ]:
# Prewitt kernels — uniform weighting (no extra centre emphasis)
prewitt_x = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]], dtype=float)

prewitt_y = np.array([[-1, -1, -1],
                      [ 0,  0,  0],
                      [ 1,  1,  1]], dtype=float)

Px = correlate(camera.astype(float), prewitt_x, mode='mirror')
Py = correlate(camera.astype(float), prewitt_y, mode='mirror')
Pmag = np.hypot(Px, Py)

# Add some Gaussian noise to see noise robustness difference
rng = np.random.default_rng(0)
noisy = np.clip(camera.astype(float) + rng.normal(0, 20, camera.shape), 0, 255).astype(np.uint8)

Smag_noisy = np.hypot(
    correlate(noisy.astype(float), sobel_x, mode='mirror'),
    correlate(noisy.astype(float), sobel_y, mode='mirror'))
Pmag_noisy = np.hypot(
    correlate(noisy.astype(float), prewitt_x, mode='mirror'),
    correlate(noisy.astype(float), prewitt_y, mode='mirror'))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
rows = [(camera,  Smag,       Pmag,       'Clean image'),
        (noisy,   Smag_noisy, Pmag_noisy, 'Noisy image (σ=20)')]
for row_axes, (orig, sm, pm, label) in zip([axes[0], axes[1]], rows):
    row_axes[0].imshow(orig, cmap='gray'); row_axes[0].set_title(f'{label} — Original'); row_axes[0].axis('off')
    row_axes[1].imshow(norm255(sm), cmap='gray'); row_axes[1].set_title('Sobel magnitude'); row_axes[1].axis('off')
    row_axes[2].imshow(norm255(pm), cmap='gray'); row_axes[2].set_title('Prewitt magnitude'); row_axes[2].axis('off')
plt.suptitle('Sobel vs Prewitt — clean vs noisy input', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 2 — Laplacian of Gaussian (LoG)

Second-derivative approach. We smooth first with a Gaussian, then apply the discrete Laplacian.
Edges appear as **zero-crossings** in the result.

In [ ]:
# Discrete Laplacian kernel
laplacian_4 = np.array([[0,  1, 0],
                         [1, -4, 1],
                         [0,  1, 0]], dtype=float)  # 4-connected

laplacian_8 = np.array([[1,  1, 1],
                         [1, -8, 1],
                         [1,  1, 1]], dtype=float)  # 8-connected

# 5×5 LoG approximation kernel (σ ≈ 1.4)
log_kernel_5x5 = np.array([[0,  0, -1,  0,  0],
                            [0, -1, -2, -1,  0],
                            [-1,-2, 16, -2, -1],
                            [0, -1, -2, -1,  0],
                            [0,  0, -1,  0,  0]], dtype=float)

# Apply LoG at different scales
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sigmas = [0.5, 1.0, 2.0, 4.0]
for i, sigma in enumerate(sigmas):
    blurred = cv2.GaussianBlur(camera, (0, 0), sigma)
    log_response = convolve(blurred.astype(float), laplacian_8, mode='reflect')
    axes[0, i].imshow(norm255(blurred), cmap='gray')
    axes[0, i].set_title(f'Gσ ∗ f  (σ={sigma})')
    axes[0, i].axis('off')
    axes[1, i].imshow(log_response, cmap='RdBu_r', vmin=-50, vmax=50)
    axes[1, i].set_title(f'∇²(Gσ ∗ f)  σ={sigma}')
    axes[1, i].axis('off')
plt.suptitle('LoG response at different scales  (red=positive, blue=negative, zero=edge)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def zero_crossings(log_img, threshold=5.0):
    """Find zero-crossings in a LoG-filtered image.
    A pixel is a zero-crossing if it has at least one 4-connected neighbour of opposite sign
    and the product |pos_max × neg_min| exceeds threshold.
    """
    edges = np.zeros(log_img.shape, dtype=bool)
    H, W = log_img.shape
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        r1 = np.clip(np.arange(H)[:, None] + dr, 0, H-1)
        c1 = np.clip(np.arange(W)[None, :] + dc, 0, W-1)
        neighbour = log_img[r1, c1]
        sign_diff = (log_img * neighbour) < 0
        strength  = np.abs(log_img - neighbour) > threshold
        edges |= (sign_diff & strength)
    return edges

sigma = 1.5
blurred_15 = cv2.GaussianBlur(camera, (0, 0), sigma)
log15 = convolve(blurred_15.astype(float), laplacian_8, mode='reflect')
zc = zero_crossings(log15, threshold=4.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(camera, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(log15, cmap='RdBu_r', vmin=-30, vmax=30); axes[1].set_title(f'LoG response (σ={sigma})'); axes[1].axis('off')
axes[2].imshow(zc, cmap='gray'); axes[2].set_title('Zero-crossings → edges'); axes[2].axis('off')
plt.suptitle('LoG edge detection via zero-crossing', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 3 — The Canny Edge Detector

We implement Canny from scratch, then compare with `cv2.Canny` to verify correctness.

In [ ]:
def non_maximum_suppression(magnitude, direction):
    """Thin edges to 1-pixel width.
    magnitude: (H, W) float, gradient magnitudes.
    direction: (H, W) float, gradient angles in radians.
    Returns: thinned magnitude array.
    """
    H, W = magnitude.shape
    suppressed = np.zeros_like(magnitude)
    # Quantise angle to 4 bins: 0, 45, 90, 135 degrees
    angle_deg = np.degrees(direction) % 180

    for i in range(1, H - 1):
        for j in range(1, W - 1):
            ang = angle_deg[i, j]
            m   = magnitude[i, j]
            if (0 <= ang < 22.5) or (157.5 <= ang <= 180):
                # Horizontal edge → compare left/right
                neighbours = (magnitude[i, j-1], magnitude[i, j+1])
            elif 22.5 <= ang < 67.5:
                # Diagonal ↗ → compare top-left / bottom-right
                #neighbours = (magnitude[i-1, j+1], magnitude[i+1, j-1])
                neighbours = (magnitude[i-1, j-1], magnitude[i+1, j+1])
            elif 67.5 <= ang < 112.5:
                # Vertical edge → compare top/bottom
                neighbours = (magnitude[i-1, j], magnitude[i+1, j])

            else: # 112.5 <= ang < 157.5
                # Diagonal ↘ → compare top-right / bottom-left
                #neighbours = (magnitude[i+1, j+1], magnitude[i-1, j-1])
                neighbours = (magnitude[i-1, j+1], magnitude[i+1, j-1])
            if m >= max(neighbours):
                suppressed[i, j] = m
    return suppressed

print('NMS function defined.')

In [ ]:
def hysteresis_thresholding(nms, t_low, t_high):
    """Double thresholding + edge tracking by hysteresis.
    Returns binary edge map.
    """
    strong = nms >= t_high
    weak   = (nms >= t_low) & (nms < t_high)

    # BFS from strong pixels, absorb connected weak pixels
    edges = strong.copy()
    from collections import deque
    queue = deque(zip(*np.where(strong)))
    visited = strong.copy()
    while queue:
        r, c = queue.popleft()
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < nms.shape[0] and 0 <= nc < nms.shape[1]:
                    if weak[nr, nc] and not visited[nr, nc]:
                        visited[nr, nc] = True
                        edges[nr, nc]   = True
                        queue.append((nr, nc))
    return edges

print('Hysteresis function defined.')

In [ ]:
def canny_from_scratch(img, sigma=1.4, t_low=20, t_high=50):
    """Complete Canny pipeline.
    Returns binary edge map (bool array).
    """
    # Step 1: Gaussian smoothing #open cv uses an internal 5x5
    blurred = cv2.GaussianBlur(img, (9, 9), sigma)

    # Step 2: Gradient magnitude and direction (Sobel)
    Gx  = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    Gy  = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.hypot(Gx, Gy)
    ang = np.arctan2(Gy, Gx)

    # Step 3: Non-maximum suppression
    nms = non_maximum_suppression(mag, ang)

    # Step 4: Double thresholding + hysteresis
    edges = hysteresis_thresholding(nms, t_low, t_high)
    return edges

print('Running Canny from scratch (this may take ~30s on a 512×512 image — NMS is pure Python)...')
edges_manual = canny_from_scratch(camera, sigma=1.4, t_low=20, t_high=50)

# Compare to cv2.Canny

#canny doesn't do the blur on its own so add as manual step
smoothed = cv2.GaussianBlur(camera, (9, 9), 1.4)
edges_cv2    = cv2.Canny(smoothed, threshold1=20, threshold2=50, L2gradient=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(camera,       cmap='gray'); axes[0].set_title('Original');             axes[0].axis('off')
axes[1].imshow(edges_manual, cmap='gray'); axes[1].set_title('Canny (from scratch)'); axes[1].axis('off')
axes[2].imshow(edges_cv2,    cmap='gray'); axes[2].set_title('cv2.Canny');            axes[2].axis('off')
plt.suptitle('Canny edge detector — manual vs OpenCV', fontsize=12)
plt.tight_layout(); plt.show()

# Rough agreement measure
agreement = np.mean(edges_manual == (edges_cv2 > 0)) * 100
print(f'Pixel agreement with cv2.Canny: {agreement:.1f}%  (expect 85–95%)')

### 3.1 — Effect of σ and Thresholds on Canny Output

In [ ]:
# σ sweep — fixed thresholds
sigmas = [0.5, 1.0, 1.4, 2.5]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sig in zip(axes, sigmas):
    e = cv2.Canny(cv2.GaussianBlur(camera, (0,0), sig), 30, 90)
    edge_frac = e.mean() * 100
    ax.imshow(e, cmap='gray')
    ax.set_title(f'σ={sig}\n{edge_frac:.1f}% edge pixels')
    ax.axis('off')
plt.suptitle('Canny: σ controls smoothing — larger σ = fewer, coarser edges', fontsize=12)
plt.tight_layout(); plt.show()

# Threshold sweep — fixed σ
threshold_pairs = [(10,30), (30,90), (50,150), (100,200)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (tlo, thi) in zip(axes, threshold_pairs):
    e = cv2.Canny(camera, tlo, thi)
    ax.imshow(e, cmap='gray')
    ax.set_title(f'T_low={tlo}, T_high={thi}')
    ax.axis('off')
plt.suptitle('Canny: threshold pair controls edge density (ratio T_high:T_low ≈ 2–3:1)', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 4 — Detector Comparison Grid

Side-by-side comparison of all four detectors on both clean and noisy images.

In [ ]:
def apply_all_detectors(img, sigma=1.4):
    """Return dict of edge maps from each detector.""",
    img_f = img.astype(float)
    blurred = cv2.GaussianBlur(img, (0, 0), sigma)

    # Sobel magnitude
    Gx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    Gy = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = norm255(np.hypot(Gx, Gy))

    # Prewitt magnitude
    px = convolve(img_f, np.array([[-1,0,1],[-1,0,1],[-1,0,1]],float), mode='reflect')
    py = convolve(img_f, np.array([[-1,-1,-1],[0,0,0],[1,1,1]],float), mode='reflect')
    prewitt_mag = norm255(np.hypot(px, py))

    # LoG zero-crossings
    log_r = convolve(blurred.astype(float), laplacian_8, mode='reflect')
    log_edges = zero_crossings(log_r, threshold=3.0).astype(np.uint8) * 255

    # Canny
    canny_edges = cv2.Canny(blurred, 30, 90)

    return {'Sobel': sobel_mag, 'Prewitt': prewitt_mag, 'LoG (zero-crossing)': log_edges, 'Canny': canny_edges}

# Clean image
rng = np.random.default_rng(7)
noisy_cam = np.clip(camera.astype(float) + rng.normal(0, 25, camera.shape), 0, 255).astype(np.uint8)

results_clean = apply_all_detectors(camera)
results_noisy = apply_all_detectors(noisy_cam)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for col, (name, em) in enumerate(results_clean.items(), start=1):
    axes[0, 0].imshow(camera,    cmap='gray'); axes[0, 0].set_title('Clean'); axes[0, 0].axis('off')
    axes[0, col].imshow(em, cmap='gray'); axes[0, col].set_title(name); axes[0, col].axis('off')
for col, (name, em) in enumerate(results_noisy.items(), start=1):
    axes[1, 0].imshow(noisy_cam, cmap='gray'); axes[1, 0].set_title('Noisy (σ=25)'); axes[1, 0].axis('off')
    axes[1, col].imshow(em, cmap='gray'); axes[1, col].set_title(name); axes[1, col].axis('off')
plt.suptitle('Detector comparison: clean (top) vs noisy input (bottom)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Canny on colour image — must convert to grayscale first
chelsea_gray_blurred = cv2.GaussianBlur(chelsea_gray, (0, 0), 1.0)
chelsea_edges = cv2.Canny(chelsea_gray_blurred, 40, 120)

# Overlay edges on original
overlay = chelsea.copy()
overlay[chelsea_edges > 0] = [255, 80, 0]   # orange edges

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(chelsea);       axes[0].set_title('Original (RGB)'); axes[0].axis('off')
axes[1].imshow(chelsea_edges, cmap='gray'); axes[1].set_title('Canny edges (grayscale)'); axes[1].axis('off')
axes[2].imshow(overlay);       axes[2].set_title('Edges overlaid on original'); axes[2].axis('off')
plt.suptitle('Applying Canny to a colour image — always convert to grayscale first', fontsize=12)
plt.tight_layout(); plt.show()

---
## Try at Home

> Complete these exercises after the lecture. They are good exam practice.

### Part A — Vectorised NMS

Our `non_maximum_suppression` above uses Python loops and is slow on large images.
Rewrite it using **NumPy slicing only** (no `for` loops over pixels).
Your implementation should produce the same output as the loop version.

In [ ]:
def nms_vectorised(magnitude, direction):
    """Vectorised non-maximum suppression.
    magnitude: (H, W) float
    direction: (H, W) float in radians
    Returns: (H, W) float — suppressed magnitude
    """
    H, W = magnitude.shape
    angle_deg = np.degrees(direction) % 180
    suppressed = np.zeros_like(magnitude)

    # Hint: use np.roll or direct indexing to shift the array
    # in each of the 4 quantised directions, then mask where
    # magnitude >= both neighbours.

    # ─── YOUR CODE HERE ───────────────────────────────────────
    raise NotImplementedError('Implement vectorised NMS')
    # ──────────────────────────────────────────────────────────

    return suppressed

# Auto-check
try:
    Gx_t = cv2.Sobel(camera, cv2.CV_64F, 1, 0, ksize=3)
    Gy_t = cv2.Sobel(camera, cv2.CV_64F, 0, 1, ksize=3)
    mag_t = np.hypot(Gx_t, Gy_t)
    ang_t = np.arctan2(Gy_t, Gx_t)
    nms_ref  = non_maximum_suppression(mag_t, ang_t)  # loop version (reference)
    nms_fast = nms_vectorised(mag_t, ang_t)
    agree    = np.mean(np.isclose(nms_ref, nms_fast, atol=1e-6)) * 100
    print(f'Agreement with loop NMS: {agree:.1f}%  (target: > 99%)')
except NotImplementedError as e:
    print(f'Not yet implemented: {e}')

### Part B — Hough Line Transform

The **Hough line transform** converts edge points into (ρ, θ) parameter space, where collinear points
accumulate into peaks — effectively finding straight lines in the image.

Using the Canny output from the cameraman image:
1. Apply `cv2.HoughLines` to find the dominant lines.
2. Draw them on the original image.
3. Tune `threshold` to keep only the strongest 10 lines.

Hint: `rho, theta = line[0]`; convert polar → Cartesian for drawing.

In [ ]:
def find_and_draw_hough_lines(img, edges, n_lines=10):
    """Find up to n_lines dominant lines using Hough transform.
    Returns BGR image with lines drawn.
    """
    result = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # ─── YOUR CODE HERE ───────────────────────────────────────
    # lines = cv2.HoughLines(edges, rho=1, theta=np.pi/180, threshold=???)
    # for line in lines[:n_lines]:
    #     rho, theta = line[0]
    #     # compute endpoints and draw with cv2.line()
    raise NotImplementedError('Implement Hough line detection')
    # ──────────────────────────────────────────────────────────

    return result

try:
    edges_cam = cv2.Canny(camera, 50, 150)
    hough_result = find_and_draw_hough_lines(camera, edges_cam, n_lines=10)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(edges_cam, cmap='gray');   axes[0].set_title('Canny edges'); axes[0].axis('off')
    axes[1].imshow(hough_result[:,:,::-1]);   axes[1].set_title('Hough lines (top 10)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()
except NotImplementedError as e:
    print(f'Not yet implemented: {e}')

---
## Recommended Reading & Resources

### Textbooks
| Source | Chapter / Section | Notes |
|--------|------------------|-------|
| **Szeliski, *Computer Vision: Algorithms and Applications* (2022)** | §7.2 | Edges and boundaries — derivations and comparisons. **Free PDF at szeliski.org/Book** |
| Gonzalez & Woods, *Digital Image Processing* (4th ed.) | §10.1–10.5 | Classic reference; detailed Canny derivation |
| Forsyth & Ponce, *Computer Vision: A Modern Approach* (2nd ed.) | §7.1–7.3 | Gradient operators and edge detection |

### Videos
| Video | Link | Duration |
|-------|------|----------|
| Computerphile — *Finding the Edges (Sobel Operator)* | youtube.com/watch?v=uihBwtPIBxM | 13 min |
| Computerphile — *Canny Edge Detector* | youtube.com/watch?v=sRFM5IEqR2w | 12 min |
| First Principles of CV — *Canny Edge Detection* | https://www.youtube.com/watch?v=hUC1uoigH6s | 6 min |
| *Convolutions in Image Processing* | youtube.com/watch?v=8rrHTtUzyZA | 36 min |

### Online References
| Source | URL |
|--------|-----|
| OpenCV Tutorial: Canny | docs.opencv.org → imgproc → tutorial_canny_detector |
| OpenCV Tutorial: Sobel | docs.opencv.org → imgproc → tutorial_sobel_derivatives |
| scikit-image edge detection | scikit-image.org/docs → api → filters (sobel, canny, roberts) |

### Original Papers
- Canny, J. (1986). *A computational approach to edge detection*. IEEE TPAMI 8(6), 679–698.
- Marr, D. & Hildreth, E. (1980). *Theory of edge detection*. Proceedings of the Royal Society, 207, 187–217.